# PUTM EV — Google Drive Connection Check

This notebook verifies:
1. Whether the Google Drive connection works
2. Whether the required folder structure exists
3. Whether the data files are in place (name, size, row count)
4. Displays a summary — what is OK and what is missing

**Required structure on Google Drive:**
```
MyDrive/
└── MOTORSPORT/
    └── PUTM_EV_FF_CurrentLoop/
        ├── data/
        │   └── model/          ← upload CSV files here
        ├── vcu_models/         ← Colab saves trained models here
        └── raports/            ← Colab saves reports and plots here
```

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")

## 1b. Diagnostics — verify Drive contents

In [ ]:
import os

root = '/content/drive/MyDrive/MOTORSPORT/PUTM_EV_FF_CurrentLoop'

print(f"Contents of {root}:")
if os.path.isdir(root):
    for item in sorted(os.listdir(root)):
        print(f"  {item}")
else:
    print("  FOLDER DOES NOT EXIST — check the name on Drive")

data_path = os.path.join(root, 'data')
if os.path.isdir(data_path):
    print(f"\nContents of data/:")
    for item in sorted(os.listdir(data_path)):
        print(f"  {item}")

    model_path = os.path.join(data_path, 'model')
    if os.path.isdir(model_path):
        print(f"\nContents of data/model/:")
        files = sorted(os.listdir(model_path))
        if files:
            for item in files:
                size_mb = os.path.getsize(os.path.join(model_path, item)) / 1024 / 1024
                print(f"  {item:<35} {size_mb:.2f} MB")
        else:
            print("  (empty)")
    else:
        print("\n  Folder data/model/ DOES NOT EXIST")
else:
    print("\n  Folder data/ DOES NOT EXIST")

## 2. Define expected structure

In [ ]:
import os

DRIVE_ROOT = '/content/drive/MyDrive/MOTORSPORT/PUTM_EV_FF_CurrentLoop'

# Required folders (identical to local structure)
REQUIRED_DIRS = {
    'data/model' : 'Training data (CSV)',
    'vcu_models' : 'Trained models (.pkl, .json)',
    'raports'    : 'Reports, plots, Excel with metrics',
}

# CSV files required in data/model/
REQUIRED_FILES = [
    'final_data.csv',
    'fsp_endu_current.csv',
    'setpoints.csv',
    'merged_wheel_data.csv',
    'front_right_data.csv',
    'rear_left_data.csv',
    'rear_right_data.csv',
]

# Minimum expected row count (excluding header)
MIN_ROWS = {
    'final_data.csv'       : 50_000,
    'fsp_endu_current.csv' : 25_000,
    'setpoints.csv'        : 50_000,
    'merged_wheel_data.csv': 15_000,
    'front_right_data.csv' : 15_000,
    'rear_left_data.csv'   : 12_000,
    'rear_right_data.csv'  : 10_000,
}

print(f"Project root: {DRIVE_ROOT}")

## 3. Check and create folders

In [ ]:
dir_status = {}

for rel_path, description in REQUIRED_DIRS.items():
    full_path = os.path.join(DRIVE_ROOT, rel_path)
    existed = os.path.isdir(full_path)
    if not existed:
        os.makedirs(full_path, exist_ok=True)
        status = 'CREATED'
    else:
        status = 'OK'
    dir_status[rel_path] = (status, full_path, description)
    icon = '✓' if existed else '⚠'
    print(f"  {icon}  [{status:8s}]  {rel_path}/  —  {description}")

print()
print("Folders ready.")

## 4. Check data files

In [ ]:
import pandas as pd

DATA_DIR = os.path.join(DRIVE_ROOT, 'data', 'model')

file_status = []

print(f"Checking files in: {DATA_DIR}\n")
print(f"  {'File':<28} {'Status':<12} {'Size':>10}  {'Rows':>10}  {'Columns'}")
print(f"  {'-'*28} {'-'*12} {'-'*10}  {'-'*10}  {'-'*30}")

for fname in REQUIRED_FILES:
    fpath = os.path.join(DATA_DIR, fname)
    row = {'file': fname}

    if not os.path.isfile(fpath):
        row.update({'status': 'MISSING', 'size': '-', 'rows': '-', 'columns': '-'})
        print(f"  ✗  {fname:<28} {'MISSING':<12} {'':>10}  {'':>10}")
    else:
        size_mb = os.path.getsize(fpath) / 1024 / 1024
        try:
            df = pd.read_csv(fpath, nrows=0)  # header only
            cols = list(df.columns)
            with open(fpath, 'r') as f:
                n_rows = sum(1 for _ in f) - 1  # -1 for header
            min_req = MIN_ROWS.get(fname, 0)
            ok = n_rows >= min_req
            status = 'OK' if ok else 'TOO FEW ROWS'
            icon = '✓' if ok else '⚠'
            row.update({'status': status, 'size': f'{size_mb:.2f} MB',
                        'rows': n_rows, 'columns': ', '.join(cols)})
            print(f"  {icon}  {fname:<28} {status:<12} {size_mb:>8.2f}MB  {n_rows:>10,}  {', '.join(cols[:6])}{'...' if len(cols)>6 else ''}")
        except Exception as e:
            row.update({'status': 'ERROR', 'size': f'{size_mb:.2f} MB', 'rows': '?', 'columns': str(e)})
            print(f"  ✗  {fname:<28} {'ERROR':<12} {size_mb:>8.2f}MB  —  {e}")

    file_status.append(row)

## 5. Check saved models (if any)

In [ ]:
import datetime

MODELS_DIR = os.path.join(DRIVE_ROOT, 'vcu_models')

model_files = [
    f for f in os.listdir(MODELS_DIR)
    if f.endswith(('.pkl', '.json', '.h5', '.pt', '.onnx'))
] if os.path.isdir(MODELS_DIR) else []

if model_files:
    print(f"Found models in {MODELS_DIR}:")
    for mf in sorted(model_files):
        mpath = os.path.join(MODELS_DIR, mf)
        size_mb = os.path.getsize(mpath) / 1024 / 1024
        mtime_str = datetime.datetime.fromtimestamp(os.path.getmtime(mpath)).strftime('%Y-%m-%d %H:%M')
        print(f"  ✓  {mf:<40} {size_mb:>8.2f} MB   saved: {mtime_str}")
else:
    print("  (no saved models — folder is empty, expected before the first training run)")

## 6. Summary

In [ ]:
missing_files = [r['file'] for r in file_status if r['status'] == 'MISSING']
broken_files  = [r['file'] for r in file_status if r['status'] not in ('OK', 'MISSING')]
ok_files      = [r['file'] for r in file_status if r['status'] == 'OK']
new_dirs      = [p for p, (s, _, _) in dir_status.items() if s == 'CREATED']

print("=" * 60)
print("  SUMMARY")
print("=" * 60)

print(f"\n  Folders:")
print(f"    OK      : {len(REQUIRED_DIRS) - len(new_dirs)}/{len(REQUIRED_DIRS)}")
if new_dirs:
    print(f"    Created : {', '.join(new_dirs)}")

print(f"\n  Data files:")
print(f"    OK      : {len(ok_files)}/{len(REQUIRED_FILES)}")

if missing_files:
    print(f"\n  MISSING FILES (upload to Drive):")
    for f in missing_files:
        print(f"    ✗  {f}")
    print(f"\n  Copy missing files to:")
    print(f"    {DATA_DIR}")

if broken_files:
    print(f"\n  FILES WITH ISSUES:")
    for f in broken_files:
        r = next(x for x in file_status if x['file'] == f)
        print(f"    ⚠  {f}  —  {r['status']}")

print()
if not missing_files and not broken_files:
    print("  ✓ All good — you can proceed to the training notebook.")
else:
    print("  ✗ Fix missing files before running training.")

print("=" * 60)
print(f"\n  Paths to use in the training notebook:")
print(f"    DATA_DIR    = '{DATA_DIR}'")
print(f"    MODELS_DIR  = '{os.path.join(DRIVE_ROOT, 'vcu_models')}'")
print(f"    REPORTS_DIR = '{os.path.join(DRIVE_ROOT, 'raports')}'")